In [5]:
# ==========================================================
# MACHINE LEARNING
# Phase 7.2 : Model Training
# ==========================================================

import os
import sys

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

print("Libraries imported successfully.")

Libraries imported successfully.


In [6]:
# ==========================================================
# PROJECT PATH
# ==========================================================

project_root = r"D:\AI-Powered-Customer-Retention-Intelligence-Platform"

os.chdir(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project Root:")
print(os.getcwd())

Project Root:
D:\AI-Powered-Customer-Retention-Intelligence-Platform


In [7]:
# ==========================================================
# LOAD MODELING DATASET
# ==========================================================

model_path = "data/processed/customer_retention_model_data.csv"

model_data = pd.read_csv(model_path)

print("=" * 60)
print("MODEL DATASET LOADED")
print("=" * 60)

print("\nShape:")
print(model_data.shape)

print("\nColumns:")
print(model_data.columns.tolist())

MODEL DATASET LOADED

Shape:
(96096, 10)

Columns:
['customer_unique_id', 'Recency', 'Frequency', 'Monetary', 'average_review_score', 'preferred_payment_method', 'average_order_value', 'customer_lifetime_value', 'Customer_Risk', 'churn_label']


In [8]:
# ==========================================================
# DEFINE FEATURES AND TARGET
# ==========================================================

feature_columns = [
    "Frequency",
    "Monetary",
    "average_review_score",
    "preferred_payment_method",
    "average_order_value"
]

target_column = "churn_label"

X = model_data[feature_columns].copy()
y = model_data[target_column].copy()

print("=" * 60)
print("FEATURES AND TARGET DEFINED")
print("=" * 60)

print("\nFeatures:")
for feature in feature_columns:
    print("-", feature)

print("\nTarget:")
print(target_column)

print("\nX Shape:", X.shape)
print("y Shape:", y.shape)

FEATURES AND TARGET DEFINED

Features:
- Frequency
- Monetary
- average_review_score
- preferred_payment_method
- average_order_value

Target:
churn_label

X Shape: (96096, 5)
y Shape: (96096,)


In [9]:
# ==========================================================
# FEATURE TYPES
# ==========================================================

numeric_features = [
    "Frequency",
    "Monetary",
    "average_review_score",
    "average_order_value"
]

categorical_features = [
    "preferred_payment_method"
]

print("Numerical Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['Frequency', 'Monetary', 'average_review_score', 'average_order_value']

Categorical Features:
['preferred_payment_method']


In [10]:
# ==========================================================
# TRAIN / TEST SPLIT
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print("\nTraining Features:", X_train.shape)
print("Testing Features :", X_test.shape)

print("\nTraining Target Distribution:")
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTesting Target Distribution:")
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

TRAIN / TEST SPLIT

Training Features: (76876, 5)
Testing Features : (19220, 5)

Training Target Distribution:
churn_label
1    71.13
0    28.87
Name: proportion, dtype: float64

Testing Target Distribution:
churn_label
1    71.13
0    28.87
Name: proportion, dtype: float64


In [11]:
# ==========================================================
# PREPROCESSING PIPELINE
# ==========================================================

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [12]:
# ==========================================================
# DEFINE MACHINE LEARNING MODELS
# ==========================================================

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
}

print("Models defined:")

for model_name in models:
    print("-", model_name)

Models defined:
- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting


In [13]:
# ==========================================================
# CREATE MODEL PIPELINES
# ==========================================================

model_pipelines = {}

for model_name, model in models.items():

    model_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "classifier",
                model
            )
        ]
    )

    model_pipelines[model_name] = model_pipeline

print("Model pipelines created successfully.")

Model pipelines created successfully.


In [14]:
# ==========================================================
# TRAIN MODELS
# ==========================================================

trained_models = {}

print("=" * 60)
print("MODEL TRAINING")
print("=" * 60)

for model_name, pipeline in model_pipelines.items():

    print(f"\nTraining: {model_name}")

    pipeline.fit(
        X_train,
        y_train
    )

    trained_models[model_name] = pipeline

    print(f"{model_name} trained successfully.")

print("\nAll models trained successfully.")

MODEL TRAINING

Training: Logistic Regression
Logistic Regression trained successfully.

Training: Decision Tree
Decision Tree trained successfully.

Training: Random Forest
Random Forest trained successfully.

Training: Gradient Boosting
Gradient Boosting trained successfully.

All models trained successfully.


In [15]:
# ==========================================================
# GENERATE TEST PREDICTIONS
# ==========================================================

predictions = {}
probabilities = {}

for model_name, pipeline in trained_models.items():

    predictions[model_name] = pipeline.predict(X_test)

    probabilities[model_name] = (
        pipeline.predict_proba(X_test)[:, 1]
    )

print("=" * 60)
print("PREDICTIONS GENERATED")
print("=" * 60)

for model_name in predictions:
    print(
        f"{model_name}: "
        f"{len(predictions[model_name])} predictions"
    )

PREDICTIONS GENERATED
Logistic Regression: 19220 predictions
Decision Tree: 19220 predictions
Random Forest: 19220 predictions
Gradient Boosting: 19220 predictions


In [16]:
# ==========================================================
# INITIAL MODEL COMPARISON
# ==========================================================

from sklearn.metrics import accuracy_score

comparison = []

for model_name in trained_models:

    accuracy = accuracy_score(
        y_test,
        predictions[model_name]
    )

    comparison.append(
        {
            "Model": model_name,
            "Accuracy": round(accuracy, 4)
        }
    )

comparison_df = pd.DataFrame(comparison)

comparison_df = comparison_df.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

print("=" * 60)
print("INITIAL MODEL COMPARISON")
print("=" * 60)

print(comparison_df)

INITIAL MODEL COMPARISON
                 Model  Accuracy
0    Gradient Boosting    0.7139
1        Random Forest    0.7127
2  Logistic Regression    0.7114
3        Decision Tree    0.7113


In [17]:
import os
import joblib

PROJECT_ROOT = os.path.abspath("..")

models_folder = os.path.join(PROJECT_ROOT, "models")
os.makedirs(models_folder, exist_ok=True)

model_path = os.path.join(
    models_folder,
    "gradient_boosting.pkl"
)

joblib.dump(
    trained_models["Gradient Boosting"],
    model_path
)

print("Model saved successfully!")
print(model_path)

Model saved successfully!
D:\models\gradient_boosting.pkl


In [18]:
print("trained_models" in globals())

True
